In [ ]:
import cv2
from ultralytics import YOLO
import torch
from IPython.display import display
import ipywidgets as widgets
import time

# ===== YOLO設定 =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = YOLO("yolov8n.pt")
model.to(device)

# ===== TCPストリーム接続（Windows → WSL）=====
url = "tcp://172.19.160.1:12345"
cap = cv2.VideoCapture(url, cv2.CAP_FFMPEG)

if not cap.isOpened():
    print("ストリーム接続失敗")
    raise RuntimeError("VideoCapture failed")

# ===== Notebook表示設定（大きく表示）=====
img_widget = widgets.Image(
    format='jpeg',
    layout=widgets.Layout(width='100%')  # 画面いっぱい
)
display(img_widget)

stop_button = widgets.Button(description="Stop")
display(stop_button)

stopped = False

def on_stop(b):
    global stopped
    stopped = True

stop_button.on_click(on_stop)

print("ストリーム開始")

# ===== メインループ =====
while not stopped:
    ret, frame = cap.read()

    if not ret:
        time.sleep(0.01)
        continue  # 一時的な受信失敗対策

    # YOLO推論（人物のみ）
    # person=0, cup=41, cell phone=67, backpack=25, handbag=31, suitcase=33
    results = model(frame, classes=[0, 41, 67, 25, 31, 33], device=device)

    annotated_frame = results[0].plot()

    # JPEGエンコード
    _, buffer = cv2.imencode('.jpeg', annotated_frame)

    img_widget.value = buffer.tobytes()

cap.release()
print("終了")